# E-Commerce Sales Analytics: Exploratory Data Analysis

This notebook explores the cleaned synthetic sample. All findings are descriptive and specific to the supplied data; they are not statistical significance tests or causal claims.

## Data Loading
Load cleaned inputs and run the reusable analysis module. Outputs are written to `data/processed/analytics/` and `reports/figures/`.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / 'src').is_dir():
    project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analytics.basic_eda import run_eda

analysis = run_eda()
customers = analysis['datasets']['customers']
products = analysis['datasets']['products']
orders = analysis['orders']
enriched_orders = analysis['enriched']
tables = analysis['tables']
kpis = analysis['kpis']
summary = analysis['summary']
print(f"Loaded {len(customers):,} customers, {len(products):,} products, and {len(orders):,} orders.")

## Data Overview

In [ ]:
import pandas as pd

overview = pd.DataFrame([
    {'dataset': name, 'rows': len(frame), 'columns': len(frame.columns)}
    for name, frame in analysis['datasets'].items()
])
overview

## Data Quality Check

In [ ]:
quality = pd.DataFrame([
    {'dataset': name, 'missing_cells': int(frame.isna().sum().sum()), 'duplicate_rows': int(frame.duplicated().sum())}
    for name, frame in analysis['datasets'].items()
])
quality

## KPI Analysis

In [ ]:
kpis.T.rename(columns={0: 'value'})

## Product Analysis
Product revenue contribution is each product's net sales divided by total net sales. The same table supports revenue and quantity rankings.

In [ ]:
top_products_by_revenue = tables['products'].nlargest(10, 'net_sales')
top_products_by_quantity = tables['products'].nlargest(10, 'quantity_sold')
bottom_products_by_revenue = tables['products'].nsmallest(10, 'net_sales')
display(top_products_by_revenue[['product_id', 'product_name', 'net_sales', 'revenue_contribution']])
display(top_products_by_quantity[['product_id', 'product_name', 'quantity_sold', 'net_sales']])
display(bottom_products_by_revenue[['product_id', 'product_name', 'net_sales', 'revenue_contribution']])

## Category Analysis
Average order value by category is net sales divided by distinct orders containing that category. Because orders currently contain one product line each, category orders are distinct order IDs in that category.

In [ ]:
display(tables['categories'][['category', 'orders', 'quantity_sold', 'net_sales', 'average_order_value', 'revenue_contribution']])
display(tables['subcategories'][['subcategory', 'orders', 'quantity_sold', 'net_sales', 'revenue_contribution']].head(10))

## Customer Analysis

In [ ]:
display(tables['customers'][['customer_id', 'customer_name', 'orders', 'net_sales']].head(10))
display(tables['customer_segments'][['customer_segment', 'orders', 'quantity_sold', 'net_sales', 'average_order_value', 'revenue_contribution']])

## Regional Analysis

In [ ]:
display(tables['customer_regions'][['customer_region', 'orders', 'quantity_sold', 'net_sales', 'average_order_value', 'revenue_contribution']])

## Time-Series Analysis
Month-over-month growth is calculated as the change in monthly net sales divided by the previous month's net sales. The first month has no prior month; growth is undefined when the prior month's net sales are zero.

In [ ]:
monthly_view = tables['monthly'].copy()
monthly_view['month'] = monthly_view['month'].dt.strftime('%Y-%m')
display(monthly_view)

## Visualizations
Each chart addresses a specific question: how net sales and orders vary by month; which categories, products, segments, and regions contribute the most net sales; and how order net sales are distributed.

In [ ]:
from IPython.display import Image, display

for figure_path in analysis['figures']:
    print(figure_path.name)
    display(Image(filename=str(figure_path)))

## Business Insights
The following statements are computed directly from the cleaned data. They describe this sample only and are not claims of statistical significance or causation.

In [ ]:
print('Highest net-sales month:', summary['major_trends']['highest_net_sales_month'])
print('Lowest net-sales month:', summary['major_trends']['lowest_net_sales_month'])
print('Months with positive month-over-month net-sales growth:', summary['major_trends']['positive_mom_growth_months'])
print('Months with negative month-over-month net-sales growth:', summary['major_trends']['negative_mom_growth_months'])
print('Top category:', summary['top_performing_categories'][0]['category'] if summary['top_performing_categories'] else 'No data')
print('Top product:', summary['top_performing_products'][0]['product_name'] if summary['top_performing_products'] else 'No data')
print(summary['interpretation_note'])